<a href="https://colab.research.google.com/github/skurakake/MMP_public/blob/main/Resnet50_C100_2026_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
BackUp_DIR = '/content/drive/MyDrive/Colab Notebooks/my_backup_4'

if not os.path.exists(BackUp_DIR):  # ディレクトリが存在しない場合、作成する。
    os.makedirs(BackUp_DIR)


In [ ]:
MODEL_DIR = '/content/drive/MyDrive/Colab Notebooks/my_bestmodel'

if not os.path.exists(MODEL_DIR):  # ディレクトリが存在しない場合、作成する。
    os.makedirs(MODEL_DIR)

In [ ]:
# cifar100 を使う
from tensorflow.keras.datasets import cifar100

(x_train, t_train), (x_test, t_test) = cifar100.load_data(label_mode='fine')


CIFAR100_LABELS_LIST = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle',
    'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel',
    'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock',
    'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
    'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster',
    'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
    'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse',
    'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear',
    'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
    'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose',
    'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake',
    'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table',
    'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout',
    'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman',
    'worm'
]

num_category = 100 # クラス数
labels = CIFAR100_LABELS_LIST

In [ ]:
# ダウンロードしたデータの一部をランダムに表示

print('x_train shape:', x_train.shape)
print('the number of train samples:', x_train.shape[0] )
print('the number of test samples:', x_test.shape[0] )
print(type(x_train[0]))

print(x_train.shape, t_train.shape)
print(x_test.shape, t_test.shape)
print(type(x_test))
print(type(t_test[0]))


num_image = 20 # 表示数
rand_idx = np.random.randint(0, len(x_train), num_image)
plt.figure(figsize=(5, 5)) # 表示領域のサイズ
# plt figure set to 5inch x 5inch(500pixel x 500 pixel).


for i in range(num_image):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_train[rand_idx[i]])
    plt.title(labels[t_train[rand_idx[i]][0]])
    plt.tick_params(labelbottom=False, labelleft=False, bottom=False, left=False)

plt.show() # 実行されたとき始めてウィンドウが立ち上がり、そのウィンドウに図が表示される
# colab では plt.show() がなくても、環境が自動でウィンドウをオープンしてくれるが、
# コマンドラインとかでは、これが必要

rand_idx = np.random.randint(0, len(x_test), num_image)
plt.figure(figsize=(5, 5)) # 表示領域のサイズ

for i in range(num_image):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_train[rand_idx[i]])
    plt.title(labels[t_train[rand_idx[i]][0]])
    plt.tick_params(labelbottom=False, labelleft=False, bottom=False, left=False)

plt.show()

In [ ]:
print("Number of training data", len(x_train))
print("Number of test data", len(x_test))
print("Number of label data for training", len(t_train))
print("Number of label data for test", len(t_test))

In [ ]:
# 設定
num_class = 100 # クラス数

# 正解データを one-hot 表現へ変換
t_train_n = tf.keras.utils.to_categorical(t_train, num_class)
t_test_n = tf.keras.utils.to_categorical(t_test, num_class)
print("正解データ（学習）：", t_train_n[5])
print("正解データ（テスト）：", t_test_n[22])

In [ ]:
x_train_N =x_train /255
x_test_N = x_test /255

In [ ]:
# 学習パラメータ等の設定
batch_size = 64 #256 # ミニバッチサイズ
epochs = 30 # エポック数

モデルの構築

Functional API を使う。
model = Model(inputs=input_tensor, outputs=outputs)　でモデルを定義する。

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Activation, Dropout, Flatten, Add
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAvgPool2D, Concatenate, AveragePooling2D
from tensorflow.keras.layers import BatchNormalization, Resizing, Rescaling
from tensorflow.keras.optimizers import Adam

#  using Functional API
def Input_stem1(input_tensor,a1):
    s1_1 = Conv2D(a1[0], a1[1], kernel_initializer='he_normal', strides=2, padding='same')(input_tensor)
    s1_2 = BatchNormalization()(s1_1)
    s1_3 = Activation('relu')(s1_2)

    return s1_3
    # input_stem1(input_tensor,(64,7))

def Input_stem2(input_tensor,a1, a2, a3):
    s2_1 = Conv2D(a1[0],a1[1], kernel_initializer='he_normal', strides=2, padding='same')(input_tensor)
    s2_2 = BatchNormalization()(s2_1)
    s2_3 = Activation('relu')(s2_2)
    s2_4 = Conv2D(a2[0],a2[1], kernel_initializer='he_normal', padding='same')(s2_3)
    s2_5 = BatchNormalization()(s2_4)
    s2_6 = Activation('relu')(s2_5)
    s2_7 = Conv2D(a3[0],a3[1], kernel_initializer='he_normal', padding='same')(s2_6)
    s2_8 = BatchNormalization()(s2_7)
    s2_9 = Activation('relu')(s2_8)

    return s2_9
    # input_stem2(input_tensor,(32,3), (32,3), (64,3))

def Input_stemNoDownSampling2(input_tensor,a1, a2, a3):
    s2_1 = Conv2D(a1[0],a1[1], kernel_initializer='he_normal', padding='same')(input_tensor)
    s2_2 = BatchNormalization()(s2_1)
    s2_3 = Activation('relu')(s2_2)
    s2_4 = Conv2D(a2[0],a2[1], kernel_initializer='he_normal', padding='same')(s2_3)
    s2_5 = BatchNormalization()(s2_4)
    s2_6 = Activation('relu')(s2_5)
    s2_7 = Conv2D(a3[0],a3[1], kernel_initializer='he_normal', padding='same')(s2_6)
    s2_8 = BatchNormalization()(s2_7)
    s2_9 = Activation('relu')(s2_8)

    return s2_9
    # input_stemNoDownSampling2(input_tensor,(32,3), (32,3), (64,3))


# Conv側の最後のBatchNormalization の初期化でガンマ＝０
def DownSamplingZeroGamma_block1(input_tensor, c1, c2, c3):
    # Path 2 is a single 1 x 1 convolutional layer
    p1_1 = Conv2D(c1[0], c1[1], kernel_initializer='he_normal', strides=2, padding='same')(input_tensor)
    p1_2 = BatchNormalization()(p1_1)
    p1_3 = Activation('relu')(p1_2)
    p1_4 = Conv2D(c2[0], c2[1], kernel_initializer='he_normal', padding='same')(p1_3)
    p1_5 = BatchNormalization()(p1_4)
    p1_6 = Activation('relu')(p1_5)
    p1_7 = Conv2D(c3[0], c3[1], kernel_initializer='he_normal', padding='same')(p1_6)
    p1_8 = BatchNormalization(gamma_initializer=tf.keras.initializers.Constant(0.0),
                              beta_initializer=tf.keras.initializers.Constant(1.0))(p1_7)
    p2_1 = Conv2D(c3[0], 1, kernel_initializer='he_normal', strides=2, padding='same')(input_tensor)
    p2_2 = BatchNormalization()(p2_1)
    p3_1 = Add()([p1_8, p2_2])
    p3_2 = Activation('relu')(p3_1)

    return p3_2


def Residual_block1(input_tensor, c1, c2, c3):
    # Path 2 is a single 1 x 1 convolutional layer
    p1_1 = Conv2D(c1[0], c1[1], kernel_initializer='he_normal', padding='same')(input_tensor)
    p1_2 = BatchNormalization()(p1_1)
    p1_3 = Activation('relu')(p1_2)
    p1_4 = Conv2D(c2[0], c2[1], kernel_initializer='he_normal', padding='same')(p1_3)
    p1_5 = BatchNormalization()(p1_4)
    p1_6 = Activation('relu')(p1_5)
    p1_7 = Conv2D(c3[0], c3[1], kernel_initializer='he_normal', padding='same')(p1_6)
    p1_8 = BatchNormalization()(p1_7)
    p2_1 = Conv2D(c3[0], 1, kernel_initializer='he_normal', padding='same')(input_tensor)
    p2_2 = BatchNormalization()(p2_1)
    p3_1 = Add()([p1_8, p2_2])
    p3_2 = Activation('relu')(p3_1)

    return p3_2

# Conv側の最後のBatchNormalization の初期化でガンマ＝０
def ResidualZeroGamma_block1(input_tensor, c1, c2, c3):
    # Path 2 is a single 1 x 1 convolutional layer
    p1_1 = Conv2D(c1[0], c1[1], kernel_initializer='he_normal', padding='same')(input_tensor)
    p1_2 = BatchNormalization()(p1_1)
    p1_3 = Activation('relu')(p1_2)
    p1_4 = Conv2D(c2[0], c2[1], kernel_initializer='he_normal', padding='same')(p1_3)
    p1_5 = BatchNormalization()(p1_4)
    p1_6 = Activation('relu')(p1_5)
    p1_7 = Conv2D(c3[0], c3[1], kernel_initializer='he_normal', padding='same')(p1_6)
    p1_8 = BatchNormalization(gamma_initializer=tf.keras.initializers.Constant(0.0),
                              beta_initializer=tf.keras.initializers.Constant(1.0))(p1_7)
    p2_1 = Conv2D(c3[0], 1, kernel_initializer='he_normal', padding='same')(input_tensor)
    p2_2 = BatchNormalization()(p2_1)
    p3_1 = Add()([p1_8, p2_2])
    p3_2 = Activation('relu')(p3_1)

    return p3_2

# strides=2 の位置が異なる
# skip 接続にaverage pooling を追懐
# Conv側の最後のBatchNormalization の初期化でガンマ＝０
def DownSampling_ZeroGamma_block(input_tensor, c1, c2, c3, c4):
    # Path 2 is a single 1 x 1 convolutional layer
    p1_1 = Conv2D(c1[0], c1[1], kernel_initializer='he_normal', padding='same')(input_tensor)
    p1_2 = BatchNormalization()(p1_1)
    p1_3 = Activation('relu')(p1_2)
    p1_4 = Conv2D(c2[0], c2[1], kernel_initializer='he_normal', strides=2, padding='same')(p1_3)
    p1_5 = BatchNormalization()(p1_4)
    p1_6 = Activation('relu')(p1_5)
    p1_7 = Conv2D(c3[0], c3[1], kernel_initializer='he_normal', padding='same')(p1_6)
    p1_8 = BatchNormalization(gamma_initializer=tf.keras.initializers.Constant(0.0),
                              beta_initializer=tf.keras.initializers.Constant(1.0))(p1_7)
    p2_1 = AveragePooling2D(c4, strides=2, padding='same')(input_tensor)
    p2_2 = Conv2D(c3[0], 1, kernel_initializer='he_normal', padding='same')(p2_1)
    p2_3 = BatchNormalization()(p2_2)
    p3_1 = Add()([p1_8, p2_3])
    p3_2 = Activation('relu')(p3_1)

    return p3_2

# strides=2 の位置が異なる
# skip 接続にaverage pooling を追懐
def DownSamplingAvePool_block2(input_tensor, c1, c2, c3, c4):
    # Path 2 is a single 1 x 1 convolutional layer
    p1_1 = Conv2D(c1[0], c1[1], kernel_initializer='he_normal', padding='same')(input_tensor)
    p1_2 = BatchNormalization()(p1_1)
    p1_3 = Activation('relu')(p1_2)
    p1_4 = Conv2D(c2[0], c2[1], kernel_initializer='he_normal', strides=2, padding='same')(p1_3)
    p1_5 = BatchNormalization()(p1_4)
    p1_6 = Activation('relu')(p1_5)
    p1_7 = Conv2D(c3[0], c3[1], kernel_initializer='he_normal', padding='same')(p1_6)
    p1_8 = BatchNormalization()(p1_7)
    p2_1 = AveragePooling2D(c4, strides=2, padding='same')(input_tensor)
    p2_2 = Conv2D(c3[0], 1, kernel_initializer='he_normal', padding='same')(p2_1)
    p2_3 = BatchNormalization()(p2_2)
    p3_1 = Add()([p1_8, p2_3])
    p3_2 = Activation('relu')(p3_1)

    return p3_2


# Define the blocks using the Functional API
def b1(input_tensor):
    x = Resizing(224,224)(input_tensor)
    #x = Input_stem1(x, (64,7))
    #x = Input_stem2(x, (32,3), (32,3), (64,3))
    x = Input_stemNoDownSampling2(input_tensor,(32,3), (32,3), (64,3)) # 画像サイズ　32*32 に合わせてstride=1 へ
    #x = MaxPooling2D(pool_size=3, strides=2, padding='same')(x)
    x = MaxPooling2D(pool_size=3, strides=1, padding='same')(x) # 画像サイズ　32*32 に合わせてstride=1 へ

    return x

def b2(input_tensor):
    #x = DownSamplingAvePool_block2(input_tensor, (64,1), (64,3), (256,1),2)
    x = Residual_block1(input_tensor, (64,1), (64,3), (256,1)) # 画像サイズ　32*32 対応
    x = ResidualZeroGamma_block1(x, (64,1), (64,3), (256,1))
    x = ResidualZeroGamma_block1(x, (64,1), (64,3), (256,1))
    return x

def b3(input_tensor):
    #x = DownSampling_ZeroGamma_block(input_tensor, (128,1), (128,3), (512,1),2)
    x = ResidualZeroGamma_block1(input_tensor, (128,1), (128,3), (512,1)) # 画像サイズ　32*32 対応
    x = ResidualZeroGamma_block1(x, (128,1), (128,3), (512,1))
    x = ResidualZeroGamma_block1(x, (128,1), (128,3), (512,1))
    x = ResidualZeroGamma_block1(x, (128,1), (128,3), (512,1))
    return x

def b4(input_tensor):
    x = DownSampling_ZeroGamma_block(input_tensor, (256,1), (256,3), (1024,1), 2)
    x = ResidualZeroGamma_block1(x, (256, 1),(256, 3), (1024,1))
    x = ResidualZeroGamma_block1(x, (256, 1),(256, 3), (1024,1))
    x = ResidualZeroGamma_block1(x, (256, 1),(256, 3), (1024,1))
    x = ResidualZeroGamma_block1(x, (256, 1),(256, 3), (1024,1))
    x = ResidualZeroGamma_block1(x, (256, 1),(256, 3), (1024,1))

    return x

def b5(input_tensor):
    x = DownSampling_ZeroGamma_block(input_tensor, (512,1), (512,3), (2048,1), 2)
    x = ResidualZeroGamma_block1(x, (512, 1),(512, 3), (2048,1))
    x = ResidualZeroGamma_block1(x, (512, 1),(512, 3), (2048,1))
    return x


# Build the complete model
input_tensor = Input(shape=x_train.shape[1:])
x = b1(input_tensor)
x = b2(x)
x = b3(x)
x = b4(x)
x = b5(x)
x = GlobalAvgPool2D()(x)

#x =Dropout(0.5)(x)
#outputs = Dense(1000)(x)
x =Dropout(0.5)(x)
outputs = Dense(num_class, kernel_initializer='he_normal', activation='softmax')(x)

model = Model(inputs=input_tensor, outputs=outputs)

initial_learning_rate = 0.0001
optimizer = Adam(learning_rate=initial_learning_rate)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.summary()

In [ ]:
# 学習レート設定コールバックを定義します。
def lr_step_decay_scheduler(epoch, lr):  # 引数 lr の最初の値は、Adam() の初期値が渡される
    warmUp_epochs=10
    lr_initial=0.0005

    if epoch <= warmUp_epochs:
        lr = lr + (lr_initial - lr) * ( epoch / warmUp_epochs )
        return lr
    elif ((epoch > warmUp_epochs) and ((epoch - warmUp_epochs)%30 ==0)):
        lr = lr/10
        return lr
    else:
        return lr


In [ ]:
# validation accuracy が更新されたら保存し、再開時にcheckpoint_cb
import json

class SaveBestCallback(tf.keras.callbacks.Callback):
    def __init__(self, filepath="best_val.json"):
        super().__init__()
        self.filepath = filepath

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        current = logs.get("val_accuracy")
        if current is None:
            return

        # 既存の best を読み込み
        try:
            with open(self.filepath, "r") as f:
                data = json.load(f)
                best = data.get("best_val_accuracy", None)
        except:
            best = None

        # 更新されていれば保存
        if best is None or current > best:
            with open(self.filepath, "w") as f:
                json.dump({"best_val_accuracy": float(current)}, f)

In [ ]:
def load_best(filepath="best_val.json"):
    try:
        with open(filepath, "r") as f:
            data = json.load(f)
            return data.get("best_val_accuracy", None)
    except:
        return None


In [ ]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(MODEL_DIR, 'best_ResNet50_2026_3.keras'),   # 保存するファイル名
    save_best_only=True,              # 検証精度が改善した時のみ保存
    monitor='val_accuracy',           # 監視する指標 (Validation Accuracy)
    mode='max',                       # 指標を最大化する方向で判断
    save_weights_only=False,     # モデルのアーキテクチャとトレーニング設定も保存する
    verbose=1                         # 保存時にログを出力
)

In [ ]:
# 学習の初回には実施しない。
# 途中中断後に再開する場合のみ、下記の「モデルを訓練する」の前に実行。
best_val = load_best()

# best 値をセット
if best_val is not None:
    checkpoint_cb.best = best_val

モデルを訓練する

In [ ]:
# callback の設定
# 学習率関数
lr_decay = tf.keras.callbacks.LearningRateScheduler(lr_step_decay_scheduler, verbose=1)
# 中断・再開用のバックアップ設定
backup_cb = tf.keras.callbacks.BackupAndRestore(BackUp_DIR)


In [ ]:
history = model.fit(x_train_N, t_train_n, batch_size=batch_size,
                    epochs=epochs, validation_data=(x_test_N, t_test_n),
                    callbacks=[lr_decay, backup_cb, checkpoint_cb, SaveBestCallback()] # ここでコールバックを指定
                    )
# model.save(BackUp_DIR + '/final_resnet_model.keras')

In [ ]:
# データの内容確認
num2_image = 20
plt.figure(figsize=(5,5))
rand_idx2 = np.random.randint(0, len(t_test), num2_image)

for i in range(num2_image):
    #x = tf.keras.layers.Resizing(96,96)(t_train[rand_idx2[i]])
    #x = tf.keras.layers.Rescaling(1.0/255)(x)
    x = x_test_N[rand_idx2[i]]
    plt.subplot(4, 5, i + 1)
    plt.imshow(x)
    #label = cifar10_labels[predicted_class[i]]
    label = CIFAR100_LABELS_LIST[t_test[rand_idx2[i]][0]]
    plt.title(label)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)

for i in range(num2_image):
    print(i, "  ", "データ番号 = ", rand_idx2[i], "   正解ラベル（テスト）：", CIFAR100_LABELS_LIST[t_test[rand_idx2[i]][0]])

plt.show()


In [ ]:
# 学習の経過をグラフで表示
import matplotlib.pyplot as plt
train_loss = history.history['loss']
train_accuracy = history.history['accuracy']
validation_loss = history.history['val_loss']
validation_accuracy = history.history['val_accuracy']

plt.plot(np.arange(len(train_loss)), train_loss, label='training_loss')
plt.plot(np.arange(len(validation_loss)), validation_loss, label='validation_loss')
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper right')
plt.show()

plt.plot(np.arange(len(train_accuracy)), train_accuracy, label='training_accuracy')
plt.plot(np.arange(len(validation_accuracy)), validation_accuracy, label='validation_accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

テストデータ全体に対して精度とlossを求める

In [ ]:
# x_test の RESIZE Rescaling が必要
loss, accuracy = model.evaluate(x_test_N, t_test_n)
print('Test loss:', loss)
print('Test accuracy:', accuracy)

学習済みのモデルを使って、テストデータの中からランダムに選択したデータに対して分類を行う。

In [ ]:
num2_image = 20
rand_idx2 = np.random.randint(0, len(t_test), num2_image)
# predictでは、ｘ_train, x_test の RESIZE Rescaling が必要
y_rand = model.predict(x_test_N[rand_idx2]) # テストデータからランダムに画像を選択して、画像分類を実行
predicted_class = np.argmax(y_rand, axis=1) # 分類結果の中で最大値を取るインデックスを取得

plt.figure(figsize=(5,5)) # 画像の表示サイズ

for i in range(num2_image):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_test[rand_idx2[i]])
    label = CIFAR100_LABELS_LIST[predicted_class[i]]
    #label = CIFAR100_LABELS_LIST[t_test[rand_idx2[i]][0]]
    plt.title(label)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)



plt.show()
